In [22]:
import html as _html
import json
import os

import folium
from folium.plugins import MarkerCluster
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

# ---------------------------------------------------------------------------
# Genre colour map
# ---------------------------------------------------------------------------
GENRE_COLORS = {
    "Drama":        "#4A90D9",
    "Thriller":     "#E05C3A",
    "Crime":        "#8B6FD4",
    "Comedy":       "#2DBD8F",
    "Action":       "#E8A030",
    "Romance":      "#D45C8A",
    "Documentary":  "#6BAD2A",
    "Horror":       "#6B5FD4",
    "Sci-Fi":       "#9B9B9B",
    "Biography":    "#2DBD8F",
    "Mystery":      "#8B6FD4",
    "Other":        "#666660",
}


def get_color(genre):
    for key in GENRE_COLORS:
        if key.lower() in str(genre).lower():
            return GENRE_COLORS[key]
    return GENRE_COLORS["Other"]

def get_sentiment_label(compound):
    if compound >=  0.75: return "Radiant"
    if compound >=  0.50: return "Cheerful"
    if compound >=  0.25: return "Uplifting"
    if compound >=  0.05: return "Warm"
    if compound >  -0.05: return "Neutral"
    if compound >= -0.25: return "Somber"
    if compound >= -0.50: return "Bleak"
    if compound >= -0.75: return "Grim"
    return "Dark"

# ---------------------------------------------------------------------------
# Popup builder
# ---------------------------------------------------------------------------
def build_popup(row):
    poster_url = str(row.get("Poster", "") or "")
    if pd.notna(row.get("Poster")) and poster_url not in ("N/A", "", "nan"):
        poster_html = (
            f'<img src="{poster_url}" '
            f'style="width:90px;min-height:120px;object-fit:cover;flex-shrink:0;">'
        )
    else:
        poster_html = (
            '<div style="width:90px;min-height:120px;background:#1a1a1a;flex-shrink:0;'
            'display:flex;align-items:center;justify-content:center;'
            'font-size:22px;color:#444;">🎬</div>'
        )

    rating_val = row.get("Imdb_rating")
    rating_str = f"{float(rating_val):.1f}" if pd.notna(rating_val) else "N/A"

    year_val = row.get("Year")
    year_str = str(int(year_val)) if pd.notna(year_val) else "N/A"

    location_str     = _html.escape(str(row.get("Locations", "N/A") or "N/A"))
    neighborhood_str = _html.escape(str(row.get("Analysis neighborhood", "N/A") or "N/A"))
    director_str     = _html.escape(str(row.get("Director", "N/A") or "N/A"))
    boxing_str       = _html.escape(str(row.get("Box_office", "N/A") or "N/A"))
    title_str        = _html.escape(str(row.get("Title", "") or ""))

    # ── Sentiment: use compound score ──────────────────────────────────────
    compound_raw = row.get("compound")
    if pd.notna(compound_raw) and compound_raw is not None:
        try:
            compound_val = float(compound_raw)
            compound_str = f"{compound_val:+.2f}"
            sent_color = (
                "#2DBD8F" if compound_val >= 0.05 else
                "#E05C3A" if compound_val <= -0.05 else
                "#888780"
            )
            sent_label = get_sentiment_label(compound_val)
            sentiment_display = f"{sent_label} ({compound_str})"
        except (ValueError, TypeError):
            sentiment_display = "N/A"
            sent_color = "#555"
            sent_label = "N/A"
            compound_str = ""
    else:
        sentiment_display = "No plot data"
        sent_color = "#555"
        sent_label = "No plot data"
        compound_str = ""

    # All-genre badges
    all_genres = row.get("All_Genres") or [row.get("Primary_Genre", "Other")]
    badges_html = ""
    for g in all_genres:
        c = get_color(g)
        badges_html += (
            f'<span style="display:inline-block;background:{c}25;'
            f'border-radius:4px;padding:2px 7px;font-size:11px;'
            f'color:{c};font-weight:500;margin:1px 2px 1px 0;">'
            f'{_html.escape(g)}</span>'
        )

    html = (
        '<div style="font-family:\'SF Pro Display\',\'Helvetica Neue\',sans-serif;'
        'font-size:12px;width:240px;'
        'background:#111111;'
        'border-radius:12px;overflow:hidden;line-height:1.5;'
        'box-shadow:0 8px 32px rgba(0,0,0,0.8);'
        'border:1px solid rgba(255,255,255,0.08);">'

        # Header
        '<div style="display:flex;align-items:stretch;">'
        + poster_html +
        '<div style="padding:10px 12px;display:flex;flex-direction:column;'
        'justify-content:center;min-width:0;gap:4px;">'
        f'<div style="font-size:13px;font-weight:600;color:#ffffff;line-height:1.3;letter-spacing:-0.01em;">{title_str}</div>'
        f'<div style="font-size:11px;color:#555;">{year_str}</div>'
        f'<div style="display:flex;flex-wrap:wrap;gap:0;">{badges_html}</div>'
        f'<div style="display:flex;align-items:center;gap:4px;font-size:12px;">'
        '<span style="font-size:12px;">⭐</span>'
        f'<span style="font-weight:600;color:#ffffff;">{rating_str}</span>'
        '<span style="color:#444;font-size:11px;">/ 10</span>'
        '</div>'
        '</div>'
        '</div>'

        # Details grid
        '<div style="border-top:1px solid rgba(255,255,255,0.06);padding:10px 12px;'
        'display:grid;grid-template-columns:auto 1fr;gap:4px 10px;'
        'font-size:11.5px;line-height:1.5;">'

        '<span style="color:#444;white-space:nowrap;">Location</span>'
        f'<span style="color:#aaa;">{location_str}</span>'

        '<span style="color:#444;white-space:nowrap;">Neighborhood</span>'
        f'<span style="color:#aaa;">{neighborhood_str}</span>'

        '<span style="color:#444;white-space:nowrap;">Director</span>'
        f'<span style="color:#aaa;">{director_str}</span>'

        '<span style="color:#444;white-space:nowrap;">Box office</span>'
        f'<span style="color:#aaa;">{boxing_str}</span>'

        '<span style="color:#444;white-space:nowrap;">Sentiment</span>'
        '<span style="display:flex;align-items:center;gap:5px;">'
        f'<span style="display:inline-block;width:6px;height:6px;border-radius:50%;'
        f'background:{sent_color};flex-shrink:0;"></span>'
        f'<span style="color:{sent_color};font-weight:500;">{_html.escape(sentiment_display)}</span>'
        '</span>'

        '</div>'
        '</div>'
    )
    return folium.Popup(html, max_width=260)

In [23]:
# ---------------------------------------------------------------------------
# Load & prepare
# ---------------------------------------------------------------------------
df = pd.read_csv("sf_movies_cleaned_og.csv")

df_map = df.dropna(subset=["Latitude", "Longitude"]).copy()
df_map["Primary_Genre"] = (
    df_map["Genres"].fillna("Other").str.split(",").str[0].str.strip()
)

# ---------------------------------------------------------------------------
# Compute sentiment scores independently
# ---------------------------------------------------------------------------

analyzer = SentimentIntensityAnalyzer()

def score_plot(text):
    if pd.isna(text) or str(text).strip() == '':
        return None
    scores = analyzer.polarity_scores(str(text))
    return round(scores['compound'], 4)

df_map['compound'] = df_map['Plot'].apply(score_plot)

print(f"Sentiment computed: {df_map['compound'].notna().sum()} films with scores, "
      f"{df_map['compound'].isna().sum()} without plot data")

Sentiment computed: 2156 films with scores, 0 without plot data


In [25]:
# ---------------------------------------------------------------------------
# Map + cluster
# ---------------------------------------------------------------------------
m1 = folium.Map(
    location=[37.7749, -122.4194],
    zoom_start=12,
    tiles="CartoDB dark_matter",
    control_scale=True,
)
cluster = MarkerCluster(
    maxClusterRadius=50,
    disableClusteringAtZoom=15,
).add_to(m1)

# ---------------------------------------------------------------------------
# Marker + record loop
# ---------------------------------------------------------------------------
records = []

for idx, (_, row) in enumerate(df_map.iterrows()):
    raw_genres = str(row.get("Genres") or "Other")
    all_genres = [g.strip() for g in raw_genres.split(",") if g.strip()]
    if not all_genres:
        all_genres = ["Other"]

    row = row.copy()
    row["All_Genres"] = all_genres

    year_val = row.get("Year")
    year     = int(year_val) if pd.notna(year_val) else None
    decade   = (year // 10 * 10) if year is not None else None

    rating_val = row.get("Imdb_rating")
    rating     = float(rating_val) if pd.notna(rating_val) else None

    box_val = None
    raw_box = str(row.get("Box_office") or "").strip()
    if raw_box and raw_box not in ("N/A", "nan"):
        try:
            box_val = int(raw_box.replace("$", "").replace(",", "").strip())
        except ValueError:
            pass

    compound_val = None
    raw_compound = row.get("compound")
    if pd.notna(raw_compound) and raw_compound is not None:
        try:
            compound_val = round(float(raw_compound), 4)
        except (ValueError, TypeError):
            pass

    has_plot = compound_val is not None

    fill_color = get_color(str(row.get("Primary_Genre", "Other")))
    marker = folium.CircleMarker(
        location=[float(row["Latitude"]), float(row["Longitude"])],
        radius=6,
        color="rgba(255,255,255,0.18)",
        weight=1.5,
        fill=True,
        fill_color=fill_color,
        fill_opacity=0.92,
        popup=build_popup(row),
    )
    marker.options["filmIdx"] = idx
    marker.add_to(cluster)

    records.append({
        "idx":      idx,
        "genres":   all_genres,
        "decade":   decade,
        "rating":   rating,
        "box":      box_val,
        "compound": compound_val,
        "has_plot": has_plot,
        "lat":      float(row["Latitude"]),
        "lng":      float(row["Longitude"]),
        "title":    str(row.get("Title", "")),
        "year":     year,
        "location": str(row.get("Locations", "") or ""),
    })

# ---------------------------------------------------------------------------
# Aggregate derivation
# ---------------------------------------------------------------------------
all_genres_flat = sorted({g for r in records for g in r["genres"] if g != "Other"})
all_decades     = sorted({r["decade"] for r in records if r["decade"] is not None})
ratings_only    = [r["rating"] for r in records if r["rating"] is not None]
boxes_only      = [r["box"]    for r in records if r["box"]    is not None]
min_rating  = min(ratings_only) if ratings_only else 0.0
max_rating  = max(ratings_only) if ratings_only else 10.0
max_box     = max(boxes_only)   if boxes_only   else 0
n_no_rating = sum(1 for r in records if r["rating"]   is None)
n_no_box    = sum(1 for r in records if r["box"]      is None)
n_no_plot   = sum(1 for r in records if not r["has_plot"])
total       = len(records)

# ---------------------------------------------------------------------------
# Build HTML fragments for the filter panel
# ---------------------------------------------------------------------------
genre_chips_html = ""
for g in all_genres_flat:
    c = get_color(g)
    genre_chips_html += (
        '<div class="chip genre-chip"'
        f' data-genre="{_html.escape(g)}" data-color="{c}"'
        ' style="display:inline-flex;align-items:center;gap:4px;cursor:pointer;'
        'padding:2px 7px;border-radius:20px;margin:1px;font-size:11px;'
        'border:1px solid rgba(255,255,255,0.08);background:rgba(255,255,255,0.04);'
        'color:#888;user-select:none;">'
        f'<span style="width:6px;height:6px;border-radius:50%;background:{c};'
        f'flex-shrink:0;"></span>{_html.escape(g)}</div>'
    )

decade_chips_html = ""
for d in all_decades:
    decade_chips_html += (
        '<div class="chip decade-chip"'
        f' data-decade="{d}"'
        ' style="display:inline-flex;align-items:center;cursor:pointer;'
        'padding:3px 9px;border-radius:20px;margin:1px;font-size:11px;'
        'border:1px solid rgba(255,255,255,0.08);background:rgba(255,255,255,0.04);'
        'color:#888;user-select:none;">'
        f'{d}s</div>'
    )

no_rating_cb = (
    '<label style="display:flex;align-items:center;gap:6px;font-size:11px;'
    'color:#444;margin-top:4px;cursor:pointer;">'
    '<input type="checkbox" id="include-no-rating" checked style="cursor:pointer;accent-color:#E8A030;">'
    f'Include unrated ({n_no_rating} films)</label>'
) if n_no_rating > 0 else ""

no_box_cb = (
    '<label style="display:flex;align-items:center;gap:6px;font-size:11px;'
    'color:#444;margin-top:4px;cursor:pointer;">'
    '<input type="checkbox" id="include-no-box" checked style="cursor:pointer;accent-color:#E8A030;">'
    f'Include unreported ({n_no_box} films)</label>'
) if n_no_box > 0 else ""

no_plot_cb = (
    '<label style="display:flex;align-items:center;gap:6px;font-size:11px;'
    'color:#444;margin-top:4px;cursor:pointer;">'
    '<input type="checkbox" id="include-no-plot" checked style="cursor:pointer;accent-color:#E8A030;">'
    f'Include no plot data ({n_no_plot} films)</label>'
) if n_no_plot > 0 else ""

records_json = json.dumps(records, ensure_ascii=False)

# ---------------------------------------------------------------------------
# Filter panel HTML
# ---------------------------------------------------------------------------
filter_panel_html = f"""
<style>
  .leaflet-popup-content-wrapper {{
    background: #111111 !important;
    border: 1px solid rgba(255,255,255,0.08) !important;
    border-radius: 12px !important;
    box-shadow: 0 8px 32px rgba(0,0,0,0.8) !important;
    padding: 0 !important;
  }}
  .leaflet-popup-content {{
    margin: 0 !important;
    width: auto !important;
  }}
  .leaflet-popup-tip {{
    background: #111111 !important;
    box-shadow: none !important;
  }}
  .leaflet-popup-close-button {{
    color: #444 !important;
    font-size: 16px !important;
    padding: 6px 8px !important;
    top: 4px !important;
    right: 4px !important;
  }}
  .leaflet-popup-close-button:hover {{
    color: #888 !important;
    background: transparent !important;
  }}
  .leaflet-container a.leaflet-popup-close-button {{
    color: #444 !important;
  }}
  #rating-min, #rating-max, #box-min {{
    -webkit-appearance: none;
    appearance: none;
    height: 3px;
    border-radius: 2px;
    background: rgba(255,255,255,0.12) !important;
    outline: none;
  }}
  #rating-min::-webkit-slider-thumb,
  #rating-max::-webkit-slider-thumb,
  #box-min::-webkit-slider-thumb {{
    -webkit-appearance: none;
    appearance: none;
    width: 13px;
    height: 13px;
    border-radius: 50%;
    background: #E8A030;
    cursor: pointer;
    border: none;
  }}
  #rating-min::-moz-range-thumb,
  #rating-max::-moz-range-thumb,
  #box-min::-moz-range-thumb {{
    width: 13px;
    height: 13px;
    border-radius: 50%;
    background: #E8A030;
    cursor: pointer;
    border: none;
  }}
  #rating-min::-moz-range-track,
  #rating-max::-moz-range-track,
  #box-min::-moz-range-track {{
    background: rgba(255,255,255,0.12);
    height: 3px;
    border-radius: 2px;
  }}
  .leaflet-bar a {{
    background-color: rgba(10,10,10,0.85) !important;
    color: #888 !important;
    border-color: rgba(255,255,255,0.08) !important;
    backdrop-filter: blur(8px);
  }}
  .leaflet-bar a:hover {{
    background-color: rgba(255,255,255,0.08) !important;
    color: #fff !important;
  }}
  .leaflet-bar {{
    border: 1px solid rgba(255,255,255,0.08) !important;
    box-shadow: 0 4px 16px rgba(0,0,0,0.6) !important;
    border-radius: 8px !important;
    overflow: hidden;
  }}
  .leaflet-control-scale-line {{
    background: rgba(10,10,10,0.75) !important;
    border-color: rgba(255,255,255,0.15) !important;
    color: #555 !important;
    font-size: 10px !important;
    backdrop-filter: blur(8px);
  }}
  .leaflet-control-attribution {{
    background: rgba(10,10,10,0.75) !important;
    color: #333 !important;
    font-size: 10px !important;
    backdrop-filter: blur(8px);
  }}
  .leaflet-control-attribution a {{
    color: #555 !important;
  }}
</style>

<!-- SF Map title banner -->
<div style="position:fixed;top:16px;left:50%;transform:translateX(-50%);
            z-index:1000;background:rgba(10,10,10,0.85);padding:8px 20px;
            border-radius:24px;border:1px solid rgba(255,255,255,0.08);
            backdrop-filter:blur(12px);-webkit-backdrop-filter:blur(12px);
            box-shadow:0 4px 24px rgba(0,0,0,0.6);
            font-family:'SF Pro Display','Helvetica Neue',sans-serif;
            font-size:13px;font-weight:600;
            color:#ffffff;pointer-events:none;white-space:nowrap;
            letter-spacing:-0.01em;">
  San Francisco Filming Locations
  <span style="font-weight:400;color:#444;">&nbsp;&middot;&nbsp;click a marker to explore</span>
</div>

<!-- Filter panel -->
<div id="filter-panel"
     style="position:fixed;top:16px;right:16px;width:272px;z-index:1000;
            background:rgba(10,10,10,0.92);
            border:1px solid rgba(255,255,255,0.08);border-radius:14px;
            box-shadow:0 8px 32px rgba(0,0,0,0.7);
            backdrop-filter:blur(16px);-webkit-backdrop-filter:blur(16px);
            font-family:'SF Pro Display','Helvetica Neue',sans-serif;">

  <!-- Header -->
  <div style="display:flex;align-items:center;justify-content:space-between;
              padding:12px 16px 10px;border-bottom:1px solid rgba(255,255,255,0.06);">
    <span style="font-size:13px;font-weight:700;color:#fff;letter-spacing:-0.01em;">Filters</span>
    <div style="display:flex;align-items:center;gap:10px;">
      <span id="match-count"
            style="font-size:11px;color:#555;background:rgba(255,255,255,0.05);
                   padding:2px 8px;border-radius:10px;border:1px solid rgba(255,255,255,0.06);">
        {total} films</span>
      <button onclick="resetFilters()"
              style="font-size:11px;color:#444;background:none;
                     border:none;cursor:pointer;padding:0;letter-spacing:0.02em;">reset</button>
    </div>
  </div>

  <!-- Filter sections -->
  <div style="padding:12px 16px;display:flex;flex-direction:column;gap:14px;
              max-height:calc(100vh - 100px);overflow-y:auto;
              box-sizing:border-box;overflow-x:hidden;">

    <!-- Genre -->
    <div>
      <div style="font-size:10px;text-transform:uppercase;letter-spacing:0.08em;
                  color:#333;font-weight:700;margin-bottom:6px;">Genre</div>
      <div id="genre-chips" style="display:flex;flex-wrap:wrap;gap:0;">
        {genre_chips_html}
      </div>
    </div>

    <!-- Decade -->
    <div>
      <div style="font-size:10px;text-transform:uppercase;letter-spacing:0.08em;
                  color:#333;font-weight:700;margin-bottom:6px;">Decade</div>
      <div id="decade-chips" style="display:flex;flex-wrap:wrap;gap:0;">
        {decade_chips_html}
      </div>
    </div>

    <!-- IMDb Rating -->
    <div style="box-sizing:border-box;">
      <div style="display:flex;justify-content:space-between;align-items:center;
                  margin-bottom:6px;">
        <div style="font-size:10px;text-transform:uppercase;letter-spacing:0.08em;
                    color:#333;font-weight:700;">IMDb Rating</div>
        <span id="rating-display" style="font-size:11px;color:#555;white-space:nowrap;margin-left:8px;"></span>
      </div>
      <div style="display:flex;gap:6px;align-items:center;width:100%;box-sizing:border-box;">
        <input type="range" id="rating-min"
               min="{min_rating}" max="{max_rating}" step="0.1" value="{min_rating}"
               style="flex:1;min-width:0;accent-color:#E8A030;">
        <input type="range" id="rating-max"
               min="{min_rating}" max="{max_rating}" step="0.1" value="{max_rating}"
               style="flex:1;min-width:0;accent-color:#E8A030;">
      </div>
      {no_rating_cb}
    </div>

    <!-- Box Office -->
    <div style="box-sizing:border-box;">
      <div style="display:flex;justify-content:space-between;align-items:center;
                  margin-bottom:6px;">
        <div style="font-size:10px;text-transform:uppercase;letter-spacing:0.08em;
                    color:#333;font-weight:700;">Box Office</div>
        <span id="box-display" style="font-size:11px;color:#555;white-space:nowrap;margin-left:8px;"></span>
      </div>
      <input type="range" id="box-min"
             min="0" max="{max_box}" step="1000000" value="0"
             style="width:100%;box-sizing:border-box;accent-color:#E8A030;">
      {no_box_cb}
    </div>

    <!-- Plot Sentiment -->
    <div>
      <div style="font-size:10px;text-transform:uppercase;letter-spacing:0.08em;
                  color:#333;font-weight:700;margin-bottom:8px;">Plot Sentiment</div>
      <div style="display:flex;gap:5px;">
        <button id="sent-btn-dark"
                onclick="setSentimentFilter('dark')"
                style="flex:1;padding:6px 8px;border-radius:20px;
                       font-size:11px;font-weight:500;cursor:pointer;
                       border:1px solid rgba(255,255,255,0.08);
                       background:rgba(255,255,255,0.04);color:#888;
                       display:inline-flex;align-items:center;justify-content:center;
                       gap:5px;transition:all 0.15s;">
          <span style="width:6px;height:6px;border-radius:50%;background:#E05C3A;flex-shrink:0;"></span>Dark</button>
        <button id="sent-btn-neutral"
                onclick="setSentimentFilter('neutral')"
                style="flex:1;padding:6px 8px;border-radius:20px;
                       font-size:11px;font-weight:500;cursor:pointer;
                       border:1px solid rgba(255,255,255,0.08);
                       background:rgba(255,255,255,0.04);color:#888;
                       display:inline-flex;align-items:center;justify-content:center;
                       gap:5px;transition:all 0.15s;">
          <span style="width:6px;height:6px;border-radius:50%;background:#666;flex-shrink:0;"></span>Neutral</button>
        <button id="sent-btn-uplifting"
                onclick="setSentimentFilter('uplifting')"
                style="flex:1;padding:6px 8px;border-radius:20px;
                       font-size:11px;font-weight:500;cursor:pointer;
                       border:1px solid rgba(255,255,255,0.08);
                       background:rgba(255,255,255,0.04);color:#888;
                       display:inline-flex;align-items:center;justify-content:center;
                       gap:5px;transition:all 0.15s;">
          <span style="width:6px;height:6px;border-radius:50%;background:#2DBD8F;flex-shrink:0;"></span>Uplifting</button>
      </div>
      {no_plot_cb}
    </div>

  </div>
</div>

<script>
(function () {{
  /* ---- data ---- */
  var RECORDS = {records_json};

  /* ---- filter state ---- */
  var activeGenres  = {{}};
  var activeDecades = {{}};
  var ratingMin = {min_rating};
  var ratingMax = {max_rating};
  var boxMin    = 0;
  var sentMin   = -1;
  var sentMax   = 1;
  var activeSentiment = null;
  var includeNoRating = true;
  var includeNoBox    = true;
  var includeNoPlot   = true;

  /* ---- marker registry ---- */
  var markerByIdx = {{}};
  var inCluster   = {{}};
  var mcgLayer    = null;

  /* ---- helpers to locate Folium objects ---- */
  function findLeafMap() {{
    for (var k in window) {{
      try {{
        if (k.indexOf('map_') === 0 &&
            window[k] != null &&
            typeof window[k].eachLayer === 'function')
          return window[k];
      }} catch (e) {{}}
    }}
    return null;
  }}

  function findCluster(leafMap) {{
    var found = null;
    leafMap.eachLayer(function (layer) {{
      if (!found &&
          typeof layer.addLayers    === 'function' &&
          typeof layer.removeLayers === 'function')
        found = layer;
    }});
    return found;
  }}

  /* ---- bootstrap ---- */
  function bootstrap() {{
    var leafMap = findLeafMap();
    if (!leafMap) {{ setTimeout(bootstrap, 200); return; }}
    mcgLayer = findCluster(leafMap);
    if (!mcgLayer) {{ setTimeout(bootstrap, 200); return; }}

    mcgLayer.eachLayer(function (marker) {{
      var idx = marker.options && marker.options.filmIdx;
      if (idx !== undefined && idx !== null) {{
        markerByIdx[idx] = marker;
        inCluster[idx]   = true;
      }}
    }});

    updateDisplays();
    applyFilters();
  }}

  /* ---- filter predicate ---- */
  function passes(rec) {{
    var gKeys = Object.keys(activeGenres);
    if (gKeys.length > 0) {{
      var hit = false;
      for (var i = 0; i < rec.genres.length; i++) {{
        if (activeGenres[rec.genres[i]]) {{ hit = true; break; }}
      }}
      if (!hit) return false;
    }}

    var dKeys = Object.keys(activeDecades);
    if (dKeys.length > 0) {{
      if (rec.decade === null || rec.decade === undefined) return false;
      if (!activeDecades[rec.decade]) return false;
    }}

    if (rec.rating === null || rec.rating === undefined) {{
      if (!includeNoRating) return false;
    }} else {{
      if (rec.rating < ratingMin || rec.rating > ratingMax) return false;
    }}

    if (rec.box === null || rec.box === undefined) {{
      if (!includeNoBox) return false;
    }} else {{
      if (rec.box < boxMin) return false;
    }}

    if (rec.compound === null || rec.compound === undefined) {{
      if (!includeNoPlot) return false;
    }} else {{
      if (rec.compound < sentMin || rec.compound > sentMax) return false;
    }}

    return true;
  }}

  /* ---- apply filters ---- */
  function applyFilters() {{
    if (!mcgLayer) return;
    var toAdd    = [];
    var toRemove = [];
    var count    = 0;
    var idxKeys  = Object.keys(markerByIdx);

    for (var i = 0; i < idxKeys.length; i++) {{
      var idx = parseInt(idxKeys[i], 10);
      var rec = RECORDS[idx];
      if (!rec) continue;
      var show = passes(rec);
      if (show) count++;
      if (show && !inCluster[idx]) {{
        toAdd.push(markerByIdx[idx]);
        inCluster[idx] = true;
      }} else if (!show && inCluster[idx]) {{
        toRemove.push(markerByIdx[idx]);
        inCluster[idx] = false;
      }}
    }}

    if (toRemove.length) mcgLayer.removeLayers(toRemove);
    if (toAdd.length)    mcgLayer.addLayers(toAdd);
    document.getElementById('match-count').textContent = count + ' films';
  }}

  /* ---- update display labels ---- */
  function updateDisplays() {{
    document.getElementById('rating-display').textContent =
      ratingMin.toFixed(1) + ' – ' + ratingMax.toFixed(1);
    var bM = Math.round(boxMin / 1e6);
    document.getElementById('box-display').textContent =
      boxMin === 0 ? '$0' : ('$' + bM + 'M+');
  }}

  /* ---- sentiment button filter ---- */
  window.setSentimentFilter = function (tone) {{
    var colors   = {{ dark: '#E05C3A', neutral: '#666666', uplifting: '#2DBD8F' }};
    var bgActive = {{ dark: 'rgba(224,92,58,0.15)', neutral: 'rgba(102,102,102,0.15)', uplifting: 'rgba(45,189,143,0.15)' }};
    var ranges   = {{
      dark:      [-1, -0.05],
      neutral:   [-0.05, 0.05],
      uplifting: [0.05, 1],
    }};

    if (activeSentiment === tone) {{
      activeSentiment = null;
      sentMin = -1;
      sentMax = 1;
    }} else {{
      activeSentiment = tone;
      sentMin = ranges[tone][0];
      sentMax = ranges[tone][1];
    }}

    ['dark', 'neutral', 'uplifting'].forEach(function (t) {{
      var btn = document.getElementById('sent-btn-' + t);
      if (!btn) return;
      if (activeSentiment === t) {{
        btn.style.background = bgActive[t];
        btn.style.color      = colors[t];
        btn.style.border     = '1px solid ' + colors[t];
      }} else {{
        btn.style.background = 'rgba(255,255,255,0.04)';
        btn.style.color      = '#888';
        btn.style.border     = '1px solid rgba(255,255,255,0.08)';
      }}
    }});

    applyFilters();
  }};

  /* ---- reset ---- */
  window.resetFilters = function () {{
    activeGenres  = {{}};
    activeDecades = {{}};
    ratingMin = {min_rating};
    ratingMax = {max_rating};
    boxMin    = 0;
    sentMin   = -1;
    sentMax   = 1;
    activeSentiment = null;
    includeNoRating = true;
    includeNoBox    = true;
    includeNoPlot   = true;

    document.querySelectorAll('.genre-chip').forEach(function (el) {{
      var c = el.getAttribute('data-color');
      el.style.background = 'rgba(255,255,255,0.04)';
      el.style.color      = '#888';
      el.style.border     = '1px solid rgba(255,255,255,0.08)';
      el.querySelector('span').style.background = c;
    }});
    document.querySelectorAll('.decade-chip').forEach(function (el) {{
      el.style.background = 'rgba(255,255,255,0.04)';
      el.style.color      = '#888';
      el.style.border     = '1px solid rgba(255,255,255,0.08)';
    }});

    ['dark', 'neutral', 'uplifting'].forEach(function (t) {{
      var btn = document.getElementById('sent-btn-' + t);
      if (btn) {{
        btn.style.background = 'rgba(255,255,255,0.04)';
        btn.style.color      = '#888';
        btn.style.border     = '1px solid rgba(255,255,255,0.08)';
      }}
    }});

    var rMinEl = document.getElementById('rating-min');
    var rMaxEl = document.getElementById('rating-max');
    if (rMinEl) rMinEl.value = {min_rating};
    if (rMaxEl) rMaxEl.value = {max_rating};

    var bMinEl = document.getElementById('box-min');
    if (bMinEl) bMinEl.value = 0;

    var cb1 = document.getElementById('include-no-rating');
    var cb2 = document.getElementById('include-no-box');
    var cb3 = document.getElementById('include-no-plot');
    if (cb1) cb1.checked = true;
    if (cb2) cb2.checked = true;
    if (cb3) cb3.checked = true;

    updateDisplays();
    applyFilters();
  }};

  /* ---- wire up UI ---- */
  window.addEventListener('load', function () {{

    /* Genre chips */
    document.querySelectorAll('.genre-chip').forEach(function (el) {{
      el.addEventListener('click', function () {{
        var g = el.getAttribute('data-genre');
        var c = el.getAttribute('data-color');
        if (activeGenres[g]) {{
          delete activeGenres[g];
          el.style.background = 'rgba(255,255,255,0.04)';
          el.style.color      = '#888';
          el.style.border     = '1px solid rgba(255,255,255,0.08)';
          el.querySelector('span').style.background = c;
        }} else {{
          activeGenres[g]     = true;
          el.style.background = c + '22';
          el.style.color      = c;
          el.style.border     = '1px solid ' + c;
          el.querySelector('span').style.background = c;
        }}
        applyFilters();
      }});
    }});

    /* Decade chips */
    document.querySelectorAll('.decade-chip').forEach(function (el) {{
      el.addEventListener('click', function () {{
        var d = parseInt(el.getAttribute('data-decade'), 10);
        if (activeDecades[d]) {{
          delete activeDecades[d];
          el.style.background = 'rgba(255,255,255,0.04)';
          el.style.color      = '#888';
          el.style.border     = '1px solid rgba(255,255,255,0.08)';
        }} else {{
          activeDecades[d]    = true;
          el.style.background = 'rgba(232,160,48,0.15)';
          el.style.color      = '#E8A030';
          el.style.border     = '1px solid #E8A030';
        }}
        applyFilters();
      }});
    }});

    /* Rating sliders */
    var rMinEl = document.getElementById('rating-min');
    var rMaxEl = document.getElementById('rating-max');
    if (rMinEl) {{
      rMinEl.addEventListener('input', function () {{
        ratingMin = parseFloat(this.value);
        if (ratingMin > ratingMax) {{ ratingMax = ratingMin; rMaxEl.value = ratingMax; }}
        updateDisplays(); applyFilters();
      }});
    }}
    if (rMaxEl) {{
      rMaxEl.addEventListener('input', function () {{
        ratingMax = parseFloat(this.value);
        if (ratingMax < ratingMin) {{ ratingMin = ratingMax; rMinEl.value = ratingMin; }}
        updateDisplays(); applyFilters();
      }});
    }}

    /* Box office slider */
    var bMinEl = document.getElementById('box-min');
    if (bMinEl) {{
      bMinEl.addEventListener('input', function () {{
        boxMin = parseInt(this.value, 10);
        updateDisplays(); applyFilters();
      }});
    }}

    /* Checkboxes */
    var cb1 = document.getElementById('include-no-rating');
    var cb2 = document.getElementById('include-no-box');
    var cb3 = document.getElementById('include-no-plot');
    if (cb1) cb1.addEventListener('change', function () {{
      includeNoRating = this.checked; applyFilters();
    }});
    if (cb2) cb2.addEventListener('change', function () {{
      includeNoBox = this.checked; applyFilters();
    }});
    if (cb3) cb3.addEventListener('change', function () {{
      includeNoPlot = this.checked; applyFilters();
    }});

    bootstrap();
  }});

}})();
</script>
"""

# ---------------------------------------------------------------------------
# Inject & save
# ---------------------------------------------------------------------------
m1.get_root().html.add_child(folium.Element(""))
m1.get_root().html.add_child(folium.Element(filter_panel_html))
m1.save("sf_map_interactive.html")
print("Saved sf_map_interactive.html")

Saved sf_map_interactive.html
